In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [4]:
df = pd.read_parquet("../data/interim/train.parquet")

with open("../config/features.json") as file:
    feature_cols = json.load(file)

X = df[feature_cols].copy()
y = df['target'].copy()

In [7]:
# tabela para iniciar as investigações
feature_profile = pd.DataFrame({
    "dtype": X.dtypes,
    "missing_count": X.isna().sum(),
    "missing_pct": X.isna().mean() * 100,
    "unique_values": X.nunique()
})

feature_profile = feature_profile.sort_values(
    "missing_pct",
    ascending = False
)

display(feature_profile)

,dtype,missing_count,missing_pct,unique_values
koi_ingress,float64,6068,100.0,0
koi_sma_err2,float64,6068,100.0,0
koi_sage,float64,6068,100.0,0
koi_eccen_err1,float64,6068,100.0,0
koi_eccen_err2,float64,6068,100.0,0
...,...,...,...,...
koi_period,float64,0,0.0,6068
koi_duration,float64,0,0.0,5443
koi_time0bk,float64,0,0.0,6055
koi_kepmag,float64,0,0.0,3152


In [ ]:
missing_summary = pd.cut(
    feature_profile["missing_pct"],
    bins=[-1, 0, 5, 25, 50, 75, 99, 100],
    labels=[
        "0%",
        "0-5%",
        "5-25%",
        "25-50%",
        "50-75%",
        "75-99%",
        "99-100%"
    ]
).value_counts(sort=False)

print("faixa de missing das variáveis elegiveis")
display(missing_summary)

missing_pct
0%          5
0-5%       62
5-25%       5
25-50%      0
50-75%      0
75-99%      0
99-100%    24
Name: count, dtype: int64

In [9]:
all_null_cols = feature_profile.query("missing_pct == 100")
constant_cols = feature_profile.query("unique_values == 1")

print("Completamente nulas:", len(all_null_cols))
print("Constantes entre valores observados:", len(constant_cols))

display(all_null_cols)
display(constant_cols)

Completamente nulas: 24
Constantes entre valores observados: 3


,dtype,missing_count,missing_pct,unique_values
koi_ingress,float64,6068,100.0,0
koi_sma_err2,float64,6068,100.0,0
koi_sage,float64,6068,100.0,0
koi_eccen_err1,float64,6068,100.0,0
koi_eccen_err2,float64,6068,100.0,0
koi_longp,float64,6068,100.0,0
koi_longp_err1,float64,6068,100.0,0
koi_longp_err2,float64,6068,100.0,0
koi_gmag_err,float64,6068,100.0,0
koi_model_dof,float64,6068,100.0,0


,dtype,missing_count,missing_pct,unique_values
koi_eccen,float64,207,3.411338,1
koi_ldm_coeff3,float64,207,3.411338,1
koi_ldm_coeff4,float64,207,3.411338,1


In [ ]:
cols = [
    "koi_eccen",
    "koi_ldm_coeff3",
    "koi_ldm_coeff4"
]

# correlação entre as variáveis constantes
X[cols].isna().astype(int).corr()

,koi_eccen,koi_ldm_coeff3,koi_ldm_coeff4
koi_eccen,1.0,1.0,1.0
koi_ldm_coeff3,1.0,1.0,1.0
koi_ldm_coeff4,1.0,1.0,1.0


In [14]:
missing_by_target = X.isna().groupby(y).mean().T * 100

missing_by_target.columns = [
    "false_positive",
    "confirmed"
]

missing_by_target["difference_pp"] = (
    missing_by_target["false_positive"]
    - missing_by_target["confirmed"]
).abs()

print("Dados faltantes de acordo com o target")
display(
    missing_by_target
    .sort_values("difference_pp", ascending=False)
    .head(15)
    .round(2)
)

Dados faltantes de acordo com o target


,false_positive,confirmed,difference_pp
koi_bin_oedp_sig,18.14,0.64,17.50
koi_max_mult_ev,13.54,0.41,13.13
koi_max_sngle_ev,13.54,0.41,13.13
koi_num_transits,13.54,0.41,13.13
koi_smass_err2,7.29,0.09,7.20
koi_srad_err2,7.29,0.09,7.20
koi_slogg_err2,7.29,0.09,7.20
koi_srad_err1,7.29,0.09,7.20
koi_smass_err1,7.29,0.09,7.20
koi_slogg_err1,7.29,0.09,7.20


In [13]:
display(X[cols].isna().astype(int).corr())

display(missing_by_target.head(3).round(2))

,koi_eccen,koi_ldm_coeff3,koi_ldm_coeff4
koi_eccen,1.0,1.0,1.0
koi_ldm_coeff3,1.0,1.0,1.0
koi_ldm_coeff4,1.0,1.0,1.0


,false_positive,confirmed,difference_pp
koi_gmag,0.78,0.05,0.73
koi_gmag_err,100.00,100.00,0.00
koi_rmag,0.13,0.00,0.13


In [15]:
# verificando se os três atributos compartilham o mesmo padrão de ausencia
cols = [
    "koi_eccen",
    "koi_ldm_coeff3",
    "koi_ldm_coeff4"
]

display(X[cols].isna().astype(int).corr())
display(missing_by_target.head(3).round(2))

print("Features completamente nulas: ", len(all_null_cols))

,koi_eccen,koi_ldm_coeff3,koi_ldm_coeff4
koi_eccen,1.0,1.0,1.0
koi_ldm_coeff3,1.0,1.0,1.0
koi_ldm_coeff4,1.0,1.0,1.0


,false_positive,confirmed,difference_pp
koi_gmag,0.78,0.05,0.73
koi_gmag_err,100.00,100.00,0.00
koi_rmag,0.13,0.00,0.13


Features completamente nulas:  24


In [16]:
display(
    missing_by_target
    .nlargest(3, "difference_pp")
    .round(2)
)

,false_positive,confirmed,difference_pp
koi_bin_oedp_sig,18.14,0.64,17.50
koi_num_transits,13.54,0.41,13.13
koi_max_sngle_ev,13.54,0.41,13.13


In [17]:
print("Tipos de dados:")
display(X.dtypes.value_counts())

print("KOIs duplicados:", df["kepoi_name"].duplicated().sum())

print("Registros completamente duplicados:", df.duplicated().sum())

print("Valores infinitos:", np.isinf(
    X.select_dtypes(include="number")
).sum().sum())

Tipos de dados:


float64    96
Name: count, dtype: int64

KOIs duplicados: 0
Registros completamente duplicados: 0
Valores infinitos: 0


## Conclusões — Qualidade dos dados

- Identificamos 24 features completamente nulas, cuja remoção será realizada durante o pré-processamento.
- Encontramos três features constantes com padrões de ausência idênticos. Recomendamos remover os atributos originais e avaliar a utilidade de um único indicador de ausência.
- Identificamos 67 features com até 5% de valores faltantes e cinco features com ausência superior a 5% e até 25%. Ambas serão inicialmente mantidas.
- Identificamos diferenças de até 17,50 pontos percentuais na proporção de valores faltantes entre as classes. Investigaremos a origem dessas ausências antes de utilizá-las como informação preditiva.

In [31]:
descriptive_stats = X.describe().T

descriptive_stats["median"] = X.median()
descriptive_stats["skewness"] = X.skew()
descriptive_stats["variance"] = X.var()

display(descriptive_stats)

,count,mean,std,min,25%,50%,75%,max,median,skewness,variance
koi_gmag,6037.0,14.809446,1.511679,7.225000e+00,13.871000,15.03100,15.912000,2.115000e+01,15.03100,-0.599161,2.285173e+00
koi_gmag_err,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
koi_rmag,6063.0,14.201172,1.389728,7.101000e+00,13.375000,14.44200,15.246500,1.996000e+01,14.44200,-0.706173,1.931343e+00
koi_rmag_err,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
koi_imag,5967.0,14.059254,1.295044,7.627000e+00,13.286000,14.28800,15.036000,1.990000e+01,14.28800,-0.648130,1.677138e+00
...,...,...,...,...,...,...,...,...,...,...,...
koi_insol_err1,5895.0,4394.361130,63974.292537,0.000000e+00,11.185000,87.21000,691.475000,3.617133e+06,87.21000,39.893001,4.092710e+09
koi_insol_err2,5895.0,-5251.306558,109138.573215,-5.600031e+06,-372.370000,-49.21000,-6.570000,0.000000e+00,-49.21000,-44.530484,1.191123e+10
koi_srho,5895.0,6.841517,48.028502,4.000000e-05,0.181770,0.78916,2.448480,9.808542e+02,0.78916,15.227974,2.306737e+03
koi_srho_err1,5895.0,15.036504,71.740407,0.000000e+00,0.033320,0.30713,1.881100,8.352423e+02,0.30713,6.894194,5.146686e+03
